# Ron-1 — Qwen3-1.7B اختبار مباشر
هذا الدفتر ينزّل أوزان Ron-1 من GitHub Release، يعيد تركيب الملف المقسّم، ثم يشغّل Qwen3-1.7B على GPU. مناسب لاختبار العقل قبل اختيار استضافة دائمة.


In [ ]:
!nvidia-smi
import os, sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.51.0', 'accelerate>=1.2.0', 'safetensors'])


In [ ]:
import os, urllib.request, json
from pathlib import Path

REPO='arkanws513-spec/Ron-1'
TAG='qwen3-1.7b-weights-v1'
API=f'https://api.github.com/repos/{REPO}/releases/tags/{TAG}'
MODEL_DIR=Path('/kaggle/working/Ron-1-Qwen3-1.7B')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

req=urllib.request.Request(API, headers={'User-Agent':'Ron-1-Kaggle-Test','Accept':'application/vnd.github+json'})
with urllib.request.urlopen(req, timeout=60) as r:
    release=json.load(r)
assets={a['name']:a['browser_download_url'] for a in release['assets']}
needed=['config.json','generation_config.json','merges.txt','model-00001-of-00002.safetensors.part-00','model-00001-of-00002.safetensors.part-01','model-00002-of-00002.safetensors','model.safetensors.index.json','tokenizer.json','tokenizer_config.json','vocab.json']
missing=[x for x in needed if x not in assets]
assert not missing, missing

def download(name):
    target=MODEL_DIR/name
    if target.exists(): return
    print('Downloading', name)
    req=urllib.request.Request(assets[name], headers={'User-Agent':'Ron-1-Kaggle-Test'})
    with urllib.request.urlopen(req, timeout=600) as r, target.open('wb') as out:
        while True:
            chunk=r.read(16*1024*1024)
            if not chunk: break
            out.write(chunk)

for n in needed: download(n)

first=MODEL_DIR/'model-00001-of-00002.safetensors'
if not first.exists():
    with first.open('wb') as out:
        for n in ['model-00001-of-00002.safetensors.part-00','model-00001-of-00002.safetensors.part-01']:
            with (MODEL_DIR/n).open('rb') as src:
                while True:
                    chunk=src.read(16*1024*1024)
                    if not chunk: break
                    out.write(chunk)
print('Model files ready:', MODEL_DIR)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer=AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
model=AutoModelForCausalLM.from_pretrained(
    str(MODEL_DIR),
    torch_dtype=torch.float16,
    device_map='auto',
    local_files_only=True,
)
print('Ron-1 Qwen3-1.7B loaded successfully.')


In [ ]:
def ron(message, max_new_tokens=256):
    messages=[
        {'role':'system','content':'أنت رون-1، مساعد عربي مستقل. أجب بوضوح ودقة وباختصار مناسب.'},
        {'role':'user','content':message},
    ]
    inputs=tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out=model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.9)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()

print('اختبار Ron-1:')
print(ron('مرحبا يا رون. من أنت؟'))
print('\nاختبار معلومات:')
print(ron('ما عاصمة مصر؟'))


## اختبار تفاعلي
شغّل الخلية التالية ثم اكتب رسائل إضافية. هذا الاختبار يشغّل الأوزان نفسها، وليس واجهة وهمية أو ردودًا محفوظة.


In [ ]:
while True:
    q=input('أنت: ').strip()
    if q.lower() in {'exit','quit','خروج'}: break
    if q: print('رون:', ron(q))
